# RAG-LIBRO — Exploración del pipeline

**Fases cubiertas:** 1a (ingestión) → 1b (FAISS) → 1c (retriever + LCEL) → **1d (`app/rag.py`)**.

Lógica de producto en `app/rag.py`; piezas en `ingest.py`, `vectorstore.py`, `llm.py`. Kernel: `.venv` de `backend/`.

In [ ]:
import sys
from pathlib import Path

BACKEND = Path.cwd()
if not (BACKEND / "app" / "ingest.py").is_file():
    BACKEND = BACKEND.parent  # si cwd es notebooks/
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

from app.ingest import (
    CHUNK_OVERLAP,
    CHUNK_SIZE,
    chunk_length_stats,
    chunks_on_pages,
    get_paths,
    ingest_pdf,
    page_pdf_1based,
    pdf_page_count,
)

## 1.1 — Scaffold: paths, `.env`, check PDF

In [ ]:
paths = get_paths()
print(f"project_root: {paths.project_root}")
print(f"pdf_path:     {paths.pdf_path}")
print(f"storage_dir:  {paths.storage_dir}")
print(f"PDF existe:   {paths.pdf_path.is_file()}")

n_pages = pdf_page_count(paths.pdf_path)
assert paths.pdf_path.is_file(), f"Colocá el PDF en {paths.pdf_path}"
print(f"Páginas PDF:  {n_pages}")

## 1.2 — `PyPDFLoader`: un `Document` por página

`metadata['page']` = 0-based (LangChain). `page_pdf` = 1-based (eval / citas).

In [ ]:
page_docs, chunk_docs = ingest_pdf(paths.pdf_path)
assert len(page_docs) == n_pages, f"{len(page_docs)} docs vs {n_pages} páginas"
print(f"Documents cargados: {len(page_docs)}")

for i in (0, 34, 179):  # primera, ~Q01, ~Q05 (0-based page 179 = PDF p.180)
    d = page_docs[i]
    print("---")
    print(f"page (0-based): {d.metadata.get('page')} | page_pdf: {page_pdf_1based(d)}")
    print(d.page_content[:400].replace("\n", " ") + "...")

## 1.3 — `RecursiveCharacterTextSplitter` (1000 / 200)

In [ ]:
stats = chunk_length_stats(chunk_docs)
print(f"chunk_size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}")
print(f"Total chunks: {stats['count']}")
print(f"Longitud (chars): min={stats['min']}, max={stats['max']}, mean={stats['mean']}")

q05 = chunks_on_pages(chunk_docs, {180, 181})
print(f"\nGate Q05 — chunks en p.180-181: {len(q05)}")
if q05:
    s = q05[0]
    print(f"  page_pdf={page_pdf_1based(s)}, len={len(s.page_content)}")
    print(f"  snippet: {s.page_content[:300].replace(chr(10), ' ')}...")

### Gate Fase 1a

- [x] PDF ~542 páginas
- [x] `len(page_docs) == n_pages`
- [x] Chunks con `page_pdf` en metadata
- [x] Al menos un chunk en p.180–181 (Q05)

**Siguiente:** Fase 1b — embeddings MiniLM + FAISS persistido.

## Fase 1b — FAISS index (bridge)

Lógica en `app/vectorstore.py`. Carga idempotente: si `storage/faiss_index/` existe, reutiliza.
Si no existe (o `REBUILD_INDEX=true`), embeddea y persiste.

In [ ]:
from app.rag import build_or_load_vectorstore
from app.vectorstore import create_embeddings, default_index_dir, index_files_exist

embeddings = create_embeddings()
vectorstore = build_or_load_vectorstore(chunk_docs, embeddings=embeddings)

print(f"Index dir:  {default_index_dir()}")
print(f"Index files: {index_files_exist(default_index_dir())}")
print(f"Vectorstore type: {type(vectorstore).__name__}")

## Fase 1c — Retriever y LCEL chain

Conectamos el índice FAISS → retriever → prompt → LLM → respuesta parseada.

### 1.7 — Retriever `k=4` + smoke test Q01 / Q05

In [ ]:
from app.ingest import page_pdf_1based
from app.eval_cases import EVAL_CASES
from app.rag import RETRIEVER_K_DEFAULT, get_retriever

RETRIEVER_K = RETRIEVER_K_DEFAULT
retriever = get_retriever(k=RETRIEVER_K, vectorstore=vectorstore)

def smoke_retrieval(query: str, expected_pages: tuple[int, ...]):
    """Recupera k docs y muestra si las páginas esperadas están en top-k."""
    docs = retriever.invoke(query)
    retrieved_pages = [page_pdf_1based(d) for d in docs]
    hit = any(p in retrieved_pages for p in expected_pages)
    print(f"Query: {query[:60]}...")
    print(f"  Expected pages: {expected_pages}")
    print(f"  Retrieved pages: {retrieved_pages}")
    print(f"  HIT: {'✓' if hit else '✗'}")
    print()
    return docs

q01 = next(c for c in EVAL_CASES if c.id == "Q01")
q05 = next(c for c in EVAL_CASES if c.id == "Q05")

docs_q01 = smoke_retrieval(q01.query, q01.expected_pages)
docs_q05 = smoke_retrieval(q05.query, q05.expected_pages)

### 1.8 — `ChatPromptTemplate` + `format_docs`

Instrucciones al LLM:
- Responder **solo** con el contexto provisto.
- Citar las páginas de donde saca la información.
- Decir "No tengo información suficiente" si el contexto no alcanza.

In [ ]:
from app.rag import RAG_PROMPT, format_docs

prompt = RAG_PROMPT

# Verificar renderizado del prompt con Q05
sample_context = format_docs(docs_q05)
rendered = prompt.format(context=sample_context, question=q05.query)
print(rendered[:800])
print(f"\n... ({len(rendered)} chars total)")

### 1.9 — Chain LCEL: retriever → prompt → LLM → parser

Operador `|` (pipe) compone Runnables. `RunnablePassthrough` pasa la query,
el retriever busca docs, `format_docs` los convierte a texto, y todo entra al prompt.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from app.llm import get_llm

llm = get_llm()

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Test con Q05
answer_q05 = rag_chain.invoke(q05.query)
print("=" * 60)
print(f"Q05: {q05.query}")
print("=" * 60)
print(answer_q05)

In [ ]:
from app.rag import answer_with_sources

# Gate 1.11–1.12: mismo contrato que test_eval.py (import desde app.rag)
result_q01 = answer_with_sources(q01.query, k=RETRIEVER_K)
print(f"Q01 pages: {result_q01['pages']}")
print(f"Q01 answer: {result_q01['answer'][:200]}...")
print()

result_q05 = answer_with_sources(q05.query, k=RETRIEVER_K)
print(f"Q05 pages: {result_q05['pages']}")
print(f"Q05 answer: {result_q05['answer'][:200]}...")

### 1.10 — Verificar `get_llm(provider=...)` desde notebook

Probamos que la cadena funciona con cada provider explícito (mock como mínimo garantizado).
El test automatizado vive en `tests/test_llm_fallback.py`.

In [ ]:
from app.llm import get_llm, Provider
from langchain_core.messages import HumanMessage

providers_to_test: list[Provider] = ["mock"]

# Agregar providers reales si tienen key configurada
import os
if os.getenv("GROQ_API_KEY", "").strip():
    providers_to_test.append("groq")
if os.getenv("OPENROUTER_API_KEY", "").strip():
    providers_to_test.append("openrouter")

print(f"Providers disponibles: {providers_to_test}\n")

test_query = "What is RAG according to this book?"
sample_context = format_docs(docs_q05[:2])

for provider in providers_to_test:
    print(f"--- Provider: {provider} ---")
    try:
        test_llm = get_llm(provider=provider)
        test_chain = (
            {"context": lambda _: sample_context, "question": RunnablePassthrough()}
            | prompt
            | test_llm
            | StrOutputParser()
        )
        out = test_chain.invoke(test_query)
        print(f"  ✓ Respuesta ({len(out)} chars): {out[:150]}...")
    except Exception as e:
        print(f"  ✗ Error: {type(e).__name__}: {e}")
    print()

### Gate Fase 1c + 1d

- [ ] Retriever `k=4` devuelve docs con `page_pdf` 1-based
- [ ] Q05: p.180–181 presente en top-4
- [ ] Prompt renderizado incluye contexto + regla "no sé"
- [ ] `answer_with_sources` desde `app.rag` devuelve `answer` + `pages`
- [ ] `get_llm(provider="mock")` funciona en la chain
- [ ] `pytest tests/test_llm_fallback.py` y `pytest tests/test_rag.py` verde

**Siguiente:** Fase 1e — `run_eval_suite` y tuning (≥70% PASS).